# 제품별 정상 분포 조건부 tail / copula

79번의 frozen COPOD-inspired / Gaussian copula를 제품별 정상 분포로 분리. CN7/RG3 정상 양상이 다르다는 공정적 가설을 검토한다. 각 외부/내부 학습 구간의 같은 제품 정상만 분포 계산. 결과를 보고 추가한 개발 실험이며 새 독립 검증 아님. 임계값은 비교 가능성을 위해 기존 내부 정상 통합 분위수 기준 유지. 두 센서 관점 최대 점수 앙상블.

https://arxiv.org/html/2009.09463v1


In [1]:
from pathlib import Path
import json,copy,numpy as np,pandas as pd,torch
from torch import nn
from sklearn.metrics import confusion_matrix
torch.set_num_threads(1)
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/74_product_normalized_joint_gas_modes.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.rsplit('memo={}',1)[0])
memo={};logs=[]
from scipy.stats import skew,norm
from sklearn.covariance import LedoitWolf
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 per=[]
 for v,(tr,te) in enumerate(views(ref,q)):
  out=np.zeros((len(q),2))
  for product in d['product'].unique():
   mask=d['product'].iloc[q].eq(product).to_numpy()
   if not mask.any():continue
   nm=(y[ref]==0)&d['product'].iloc[ref].eq(product).to_numpy()
   normal=tr[nm];sort=np.sort(normal,axis=0);n=len(normal)
   assert n>=20
   asym=np.sign(np.nan_to_num(skew(normal,axis=0)))
   def ranks(x):
    l=np.column_stack([(np.searchsorted(sort[:,j],x[:,j],side='right')+1)/(n+2) for j in range(x.shape[1])])
    r=np.column_stack([(n-np.searchsorted(sort[:,j],x[:,j],side='left')+1)/(n+2) for j in range(x.shape[1])])
    return l,r
   def tail(x):
    l,r=ranks(x);a=-np.log(l);b=-np.log(r)
    return np.maximum(np.where(asym>0,b,np.where(asym<0,a,a+b)),(a+b)/2).sum(1)
   def gauss(x):
    l,r=ranks(x);return norm.ppf(np.clip((l+1-r)/2,1/(n+2),1-1/(n+2)))
   cov=LedoitWolf().fit(gauss(normal))
   train=np.column_stack([tail(normal),cov.mahalanobis(gauss(normal))])
   scores=np.column_stack([tail(te[mask]),cov.mahalanobis(gauss(te[mask]))])
   out[mask]=scores/np.maximum(np.quantile(train,.95,axis=0),1e-6)
   logs.append(dict(train=len(ref),query=int(mask.sum()),product=product,view=v,normal_training=n,gas_training=0,train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
  per.append(out)
 memo[key]=np.max(per,axis=0)
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 outer=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['product_frozen_tail','product_Gaussian_copula']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,outer[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(outer[j,k]>cut),score=float(outer[j,k]),threshold=cut))
 selected=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j]),score=np.nan,threshold=np.nan))
 print('Finished Product copula block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(logs),'training_audit')]:
 obj.to_csv(tables/f'gas_product_copula_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



CN7 extra metadata 713 sensor-consistent unique inputs 713 conflicting 0


Finished Product copula block 1


Finished Product copula block 2


Finished Product copula block 3


Finished Product copula block 4


...

Finished Product copula block 7


Finished Product copula block 8


Finished Product copula block 9


Finished Product copula block 10


                      model  budget product  TP  FP  FN  TN   recall       F1
inner_recall_first_selected    0.00     CN7   6  92   4 603 0.600000 0.111111
inner_recall_first_selected    0.00     RG3   2  41  17 560 0.105263 0.064516
    product_Gaussian_copula    0.02     CN7   2  15   8 680 0.200000 0.148148
    product_Gaussian_copula    0.02     RG3   1  12  18 589 0.052632 0.062500
    product_Gaussian_copula    0.05     CN7   4  59   6 636 0.400000 0.109589
    product_Gaussian_copula    0.05     RG3   1  18  18 583 0.052632 0.052632
    product_Gaussian_copula    0.10     CN7   6  85   4 610 0.600000 0.118812
    product_Gaussian_copula    0.10     RG3   4  39  15 562 0.210526 0.129032
        product_frozen_tail    0.02     CN7   2  36   8 659 0.200000 0.083333
        product_frozen_tail    0.02     RG3   0   1  19 600 0.000000 0.000000
        product_frozen_tail    0.05     CN7   5  59   5 636 0.500000 0.135135
        product_frozen_tail    0.05     RG3   0  12  19 589 0.00